In [33]:
import pandas as pd
from datasets import Dataset

In [20]:
df = pd.read_json('../checkFieldPredictions.json')

df = df.rename(columns={'predictedValue': 'value'}).drop(columns=['id', 'modelName', 'modelVersion', 'updatedAt'])

print(df.shape)
print(df['checkDetectionResultId'].nunique())

df.head()

(101260, 5)
9057


,checkDetectionResultId,field,value,confidence,createdAt
0,b25dbf83-00f4-43d1-ab11-1920d4a972ab,raw_date,02/25/2025,99.0,2025-03-05 21:07:58
1,4daac08c-8973-41e4-8201-4feac4adf0e4,raw_routing_number,102100400,99.0,2025-03-13 22:48:20
2,f8ca6e66-7e7f-41e4-b3c3-e0188865f1ad,normalized_amount_numeric,739.75,99.0,2025-02-28 02:08:02
3,3fdcd832-f5e8-479a-933f-51f6a75b00fc,raw_payer,Western Union,64.0,2025-03-07 00:17:48
4,4dd80039-dd6a-4ba2-a02c-cfe6e3c5e163,normalized_amount_numeric,322.75,98.0,2025-03-12 22:55:34


In [21]:
df['field'].unique()

array(['raw_date', 'raw_routing_number', 'normalized_amount_numeric',
       'raw_payer', 'raw_bank_name', 'normalized_payee',
       'raw_amount_words', 'raw_amount_numeric', 'raw_account_number',
       'normalized_check_number', 'validated_amount_numeric', 'raw_payee',
       'normalized_payer', 'raw_check_number', 'raw_memo'], dtype=object)

In [22]:
# filter out fields that are not normalized_amount_numeric or raw_amount_words
df = df[(df["field"] == "normalized_amount_numeric") | (df["field"] == "raw_amount_words")]

print(df.shape)
print(df['checkDetectionResultId'].nunique())

df.head()

(14914, 5)
7973


,checkDetectionResultId,field,value,confidence,createdAt
2,f8ca6e66-7e7f-41e4-b3c3-e0188865f1ad,normalized_amount_numeric,739.75,99.0,2025-02-28 02:08:02
4,4dd80039-dd6a-4ba2-a02c-cfe6e3c5e163,normalized_amount_numeric,322.75,98.0,2025-03-12 22:55:34
8,f1fdee37-b171-491c-b45a-7b4f12405db8,raw_amount_words,Seven Hundred Fourteen and 48/100 Dollars,99.0,2025-03-04 16:47:07
13,58f3cb33-70b5-4a64-9ee3-ad88d3eb3b63,raw_amount_words,THREE HUNDRED AND 00/100 DOLLARS,98.0,2025-03-13 15:06:27
18,31135c50-6494-45df-9688-a358f4657a8c,raw_amount_words,One thousand twenty five dollars and 08 cents,97.0,2025-03-16 19:06:20


In [30]:
# pivot the table
# i want to have one row per checkDetectionResultId
# and for columns i want checkDetectionResultId, normalized_amount_numeric_value, normalized_amount_numeric_confidence, raw_amount_string_value, raw_amount_string_confidence
# Pivot the table
pivoted_df = df.pivot(index="checkDetectionResultId", columns="field", values=["value", "confidence"])

# Flatten the MultiIndex columns
pivoted_df.columns = ["_".join(col).strip() for col in pivoted_df.columns.values]

# Reset index to make checkDetectionResultId a column
pivoted_df = pivoted_df.reset_index()

# Rename columns for clarity
pivoted_df = pivoted_df.rename(
    columns={
        "value_normalized_amount_numeric": "numeric",
        "confidence_normalized_amount_numeric": "numeric_confidence",
        "value_raw_amount_words": "words",
        "confidence_raw_amount_words": "words_confidence",
    }
)

print(pivoted_df)

                    checkDetectionResultId   numeric  \
0     000c421d-528f-45fe-888e-de46f833d2f1   1365.61   
1     003abd33-4c2b-438c-8a66-9da35761ab23    394.00   
2     00401eca-cee6-4832-b0ea-b3039b27d3fe    964.00   
3     00443cb7-8c71-4948-ba2c-bb3555737f88    208.75   
4     00576201-6ff3-43f7-9215-0898c0a93c40     71.83   
...                                    ...       ...   
7968  ffcd2605-d738-4806-8da3-d865931bc989  24893.32   
7969  ffd7ee5a-e544-4a1c-bd31-4d0006b855e1       NaN   
7970  ffdc7d01-919a-4b07-9e61-c5d7db0f7687  33736.86   
7971  ffdf1aff-3e79-4960-9b9d-6a520fcaa70a    500.00   
7972  ffed8662-e3d4-4694-b1c3-72dfacfdc2a6    165.80   

                                                  words numeric_confidence  \
0     **One Thousand Three Hundred Sixty Five Dollar...               98.0   
1                  Three Hundred Ninety-Four and 00/100               99.0   
2            NINE HUNDRED SIXTY FOUR AND 00/100 DOLLARS               87.0   
3      Two Hund

In [31]:
pivoted_df.head()

,checkDetectionResultId,numeric,words,numeric_confidence,words_confidence
0,000c421d-528f-45fe-888e-de46f833d2f1,1365.61,**One Thousand Three Hundred Sixty Five Dollar...,98.0,92.0
1,003abd33-4c2b-438c-8a66-9da35761ab23,394.00,Three Hundred Ninety-Four and 00/100,99.0,99.0
2,00401eca-cee6-4832-b0ea-b3039b27d3fe,964.00,NINE HUNDRED SIXTY FOUR AND 00/100 DOLLARS,87.0,94.0
3,00443cb7-8c71-4948-ba2c-bb3555737f88,208.75,Two Hundred Eight Dollars And Seventy Five Cents,98.0,98.0
4,00576201-6ff3-43f7-9215-0898c0a93c40,71.83,Seventy-One and 83/100,99.0,95.0


In [34]:
dataset = Dataset.from_pandas(pivoted_df)

dataset = dataset.train_test_split(test_size=0.5)

dataset

DatasetDict({
    train: Dataset({
        features: ['checkDetectionResultId', 'numeric', 'words', 'numeric_confidence', 'words_confidence'],
        num_rows: 3986
    })
    test: Dataset({
        features: ['checkDetectionResultId', 'numeric', 'words', 'numeric_confidence', 'words_confidence'],
        num_rows: 3987
    })
})

In [37]:
dataset.save_to_disk('../gold-dataset')

Saving the dataset (0/1 shards):   0%|          | 0/3986 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/3987 [00:00<?, ? examples/s]